# Impacts on Global Oil Trade

This notebook examines whether Singapore's oil trade shifted away from Middle East supply and toward alternative sources after geopolitical tensions around the Strait of Hormuz.

**Americas** is used as a proxy for US-linked supply, and **Africa** is used as a proxy for Cape of Good Hope / Africa-linked alternative routes.


## 1. Load Libraries

In [21]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.float_format = "{:,.2f}".format

## 2. Load Singapore Oil Import Data

In [22]:
# Detect project root from either the repository root or the Visualisation_Board folder.
cwd = Path.cwd()
if (cwd / "Data" / "raw").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "Data" / "raw").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path("/Users/veronica/Desktop/AI-Immigration-main")

RAW = PROJECT_ROOT / "Data" / "raw"

imports = pd.read_parquet(RAW / "singapore_imports_20260603_112515.parquet")
imports["date"] = pd.to_datetime(imports["date"])

imports.head()


,date,country,import_value
0,2019-01-01,Middle East,62.50
1,2019-02-01,Middle East,61.79
2,2019-03-01,Middle East,62.51
3,2019-04-01,Middle East,63.31
4,2019-05-01,Middle East,61.49


## 3. Descriptive Statistics

In [23]:
imports.groupby("country")["import_value"].describe()


,count,mean,std,min,25%,50%,75%,max
country,,,,,,,,
Africa,84.00,6.69,1.05,4.72,5.79,6.78,7.56,8.91
Americas,84.00,8.92,1.83,5.60,7.26,8.93,10.46,12.76
Middle East,84.00,58.99,1.91,54.20,57.59,58.95,60.31,63.31
Other Asia-Pacific,84.00,25.40,1.62,22.32,24.06,25.52,26.56,28.61


## 4. Main Visualization: Middle East vs Alternative Supply

In [24]:
alt_imports = (
    imports
    .assign(
        group=lambda df: np.where(
            df["country"].isin(["Americas", "Africa"]),
            "Americas + Africa alternative supply",
            df["country"],
        )
    )
    .groupby(["date", "group"], as_index=False)["import_value"]
    .sum()
)

focus = alt_imports[
    alt_imports["group"].isin([
        "Middle East",
        "Americas + Africa alternative supply",
    ])
]

fig = px.line(
    focus,
    x="date",
    y="import_value",
    color="group",
    markers=True,
    title="Singapore Oil Imports: Middle East vs Alternative Supply Proxy",
    labels={
        "date": "Month",
        "import_value": "Import share / index",
        "group": "Supply source",
    },
)

fig.update_layout(template="plotly_white", hovermode="x unified")
fig.show()


## Microanalysis 1 : Singapore Oil Imports: Ukraine War Period (2021–2022)

This chart examines how Singapore's oil import composition shifted during the
Russia-Ukraine war, focusing on **June 2022** — the point at which Western
sanctions on Russian oil began to fully take effect and global supply routes
were meaningfully disrupted.

Two supply sources are tracked:
- **Middle East** — historically Singapore's dominant oil supplier (~59–62% share)
- **Americas + Africa** — a proxy for alternative, non-Russian supply diversification

The dual y-axis design allows both series to be read at their own scale,
making relative changes visible even when absolute levels differ greatly.


In [25]:
# ── Ukraine War Window: Jun 2022 Focus ────────────────────────────────────────
war_window = focus[
    (focus["date"] >= "2021-01-01") &
    (focus["date"] <= "2022-12-31")
].copy()

# Helper: convert date string to milliseconds (required by Plotly for vline/vrect)
def to_ms(date_str):
    return pd.Timestamp(date_str).timestamp() * 1000

# Separate the two series
me = war_window[war_window["group"] == "Middle East"].sort_values("date")
am = war_window[war_window["group"] == "Americas + Africa alternative supply"].sort_values("date")

# Compute pre/post means around Jun 2022
def delta(df, col="import_value", pre_end="2022-05-31", post_start="2022-06-01"):
    pre  = df[df["date"] <= pre_end][col].mean()
    post = df[df["date"] >= post_start][col].mean()
    return pre, post, post - pre, (post - pre) / pre * 100

me_pre, me_post, me_d, me_pct = delta(me)
am_pre, am_post, am_d, am_pct = delta(am)

# Build annotation text as single-line strings (avoids f-string syntax errors)
me_label = f"Middle East | Pre: {me_pre:.1f} → Post: {me_post:.1f} | Δ {me_d:+.1f} ({me_pct:+.1f}%)"
am_label = f"Americas + Africa | Pre: {am_pre:.1f} → Post: {am_post:.1f} | Δ {am_d:+.1f} ({am_pct:+.1f}%)"

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Dual y-axis: each series gets its own scale so small changes are visible
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=me["date"], y=me["import_value"],
    name="Middle East", mode="lines+markers",
    line=dict(color="#D84040", width=2.5),
    marker=dict(size=5),
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=am["date"], y=am["import_value"],
    name="Americas + Africa", mode="lines+markers",
    line=dict(color="#3B82F6", width=2.5),
    marker=dict(size=5),
), secondary_y=True)

# Shaded region around Jun 2022 and dashed event line
fig.add_vrect(
    x0=to_ms("2022-05-15"), x1=to_ms("2022-07-15"),
    fillcolor="rgba(255,200,0,0.15)", line_width=0,
)
fig.add_vline(
    x=to_ms("2022-06-01"),
    line_dash="dash", line_color="orange", line_width=2,
)

# Event label at top — clear of both data series
fig.add_annotation(
    x=to_ms("2022-06-01"), y=1.07,
    xref="x", yref="paper",
    text="Jun 2022 shift", showarrow=False,
    font=dict(size=11, color="darkorange"),
    bgcolor="rgba(255,200,0,0.2)",
)

# Middle East annotation — pushed to top-left corner with arrow pointing to data
fig.add_annotation(
    x=to_ms("2022-06-01"), y=me_post,
    xref="x", yref="y",
    text=me_label,
    showarrow=True, arrowhead=2,
    arrowcolor="#D84040",
    ax=-280, ay=-120,
    bgcolor="rgba(255,255,255,0.92)",
    bordercolor="#D84040", borderwidth=1,
    font=dict(size=10, color="#D84040"),
    align="left",
)

# Americas annotation — pushed to bottom-right corner with arrow pointing to data
fig.add_annotation(
    x=to_ms("2022-06-01"), y=am_post,
    xref="x", yref="y2",
    text=am_label,
    showarrow=True, arrowhead=2,
    arrowcolor="#3B82F6",
    ax=280, ay=120,
    bgcolor="rgba(255,255,255,0.92)",
    bordercolor="#3B82F6", borderwidth=1,
    font=dict(size=10, color="#3B82F6"),
    align="left",
)

# Axis labels coloured to match their respective series
fig.update_yaxes(
    title_text="Middle East import share", secondary_y=False,
    title_font=dict(color="#D84040"), tickfont=dict(color="#D84040"),
)
fig.update_yaxes(
    title_text="Americas + Africa import share", secondary_y=True,
    title_font=dict(color="#3B82F6"), tickfont=dict(color="#3B82F6"),
)

fig.update_layout(
    title="Singapore Oil Imports: Ukraine War Window — Jun 2022 Focus",
    xaxis_title="Month",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80),
)
fig.show()

## Microanalysis 2 Singapore Oil Imports: Iran Conflict Period (2024–2026)

This chart tracks Singapore's oil import composition from April 2024 onwards,
coinciding with the escalation of the Iran-Israel conflict. Two key events are
marked:

- **14 April 2024** — Iran launched an unprecedented direct drone and missile
  strike on Israeli territory, the first such attack in the two countries'
  history, raising fears of a broader regional war and potential disruption to
  Strait of Hormuz oil flows.
- **19 April 2024** — Israel carried out a limited counter-strike inside Iran,
  further heightening market uncertainty around Middle Eastern supply security.

The **dashed trend lines** show 3-month rolling averages for each supply source,
smoothing out month-to-month noise to reveal the underlying directional shift.
This is particularly useful here as monthly tanker arrival data can be lumpy.

In [29]:
# ── Cell B: Iran conflict trend (Apr 2024 – latest) ──────────────────────────
from plotly.subplots import make_subplots
import plotly.graph_objects as go

iran_window = focus[focus["date"] >= "2024-01-01"].copy()

def to_ms(date_str):
    return pd.Timestamp(date_str).timestamp() * 1000

# Separate + sort series
me = iran_window[iran_window["group"] == "Middle East"].sort_values("date")
am = iran_window[iran_window["group"] == "Americas + Africa alternative supply"].sort_values("date")

# Rolling 3-month averages
me = me.copy(); me["rolling"] = me["import_value"].rolling(3, min_periods=1).mean()
am = am.copy(); am["rolling"] = am["import_value"].rolling(3, min_periods=1).mean()

# Post-escalation deltas
am_pre  = am[am["date"] <  "2024-04-14"]["import_value"].mean()
am_post = am[am["date"] >= "2024-04-14"]["import_value"].mean()
am_d    = am_post - am_pre
am_pct  = am_d / am_pre * 100

me_pre  = me[me["date"] <  "2024-04-14"]["import_value"].mean()
me_post = me[me["date"] >= "2024-04-14"]["import_value"].mean()
me_d    = me_post - me_pre
me_pct  = me_d / me_pre * 100

# ── Dual y-axis figure ────────────────────────────────────────────────────────
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Middle East raw (faded) + trend (bold)
fig.add_trace(go.Scatter(
    x=me["date"], y=me["import_value"],
    name="Middle East", mode="lines+markers",
    line=dict(color="#D84040", width=1.5),
    marker=dict(size=4), opacity=0.35,
    showlegend=True,
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=me["date"], y=me["rolling"],
    name="Middle East 3-mo avg", mode="lines",
    line=dict(color="#D84040", width=3, dash="dash"),
), secondary_y=False)

# Americas raw (faded) + trend (bold)
fig.add_trace(go.Scatter(
    x=am["date"], y=am["import_value"],
    name="Americas + Africa", mode="lines+markers",
    line=dict(color="#3B82F6", width=1.5),
    marker=dict(size=4), opacity=0.35,
    showlegend=True,
), secondary_y=True)

fig.add_trace(go.Scatter(
    x=am["date"], y=am["rolling"],
    name="Americas + Africa 3-mo avg", mode="lines",
    line=dict(color="#3B82F6", width=3, dash="dash"),
), secondary_y=True)

# ── Escalation shading + event lines ─────────────────────────────────────────
fig.add_vrect(
    x0=to_ms("2024-04-14"), x1=to_ms("2024-04-30"),
    fillcolor="rgba(255,165,0,0.12)", line_width=0,
)
fig.add_vline(x=to_ms("2024-04-14"), line_dash="dot",
              line_color="orange", line_width=1.5)
fig.add_vline(x=to_ms("2024-04-19"), line_dash="dot",
              line_color="darkorange", line_width=1.5)

# ── Event labels — placed BELOW the chart to avoid title overlap ──────────────
fig.add_annotation(
    x=to_ms("2024-04-14"), y=-0.18, xref="x", yref="paper",
    text="① Iran drone strike on Israel (Apr 14)",
    showarrow=False,
    font=dict(size=10, color="orange"),
    bgcolor="rgba(255,255,255,0.0)",
    xanchor="left",
)
fig.add_annotation(
    x=to_ms("2024-04-14"), y=-0.24, xref="x", yref="paper",
    text="② Israeli counter-strike (Apr 19)",
    showarrow=False,
    font=dict(size=10, color="darkorange"),
    bgcolor="rgba(255,255,255,0.0)",
    xanchor="left",
)

# ── Delta callouts anchored to end of each trend line ────────────────────────
am_latest = am.iloc[-1]
fig.add_annotation(
    x=am_latest["date"], y=am_latest["rolling"],
    xref="x", yref="y2",
    text=f"Δ {am_d:+.1f} ({am_pct:+.1f}%) since escalation",
    showarrow=True, arrowhead=2, arrowcolor="#3B82F6",
    ax=-180, ay=-45,
    bgcolor="rgba(255,255,255,0.92)",
    bordercolor="#3B82F6", borderwidth=1,
    font=dict(size=10, color="#3B82F6"),
)

me_latest = me.iloc[-1]
fig.add_annotation(
    x=me_latest["date"], y=me_latest["rolling"],
    xref="x", yref="y",
    text=f"Δ {me_d:+.1f} ({me_pct:+.1f}%) since escalation",
    showarrow=True, arrowhead=2, arrowcolor="#D84040",
    ax=-180, ay=45,
    bgcolor="rgba(255,255,255,0.92)",
    bordercolor="#D84040", borderwidth=1,
    font=dict(size=10, color="#D84040"),
)

# ── Axes ──────────────────────────────────────────────────────────────────────
fig.update_yaxes(
    title_text="Middle East import share",
    secondary_y=False,
    title_font=dict(color="#D84040"),
    tickfont=dict(color="#D84040"),
)
fig.update_yaxes(
    title_text="Americas + Africa import share",
    secondary_y=True,
    title_font=dict(color="#3B82F6"),
    tickfont=dict(color="#3B82F6"),
)

fig.update_layout(
    title=dict(
        text="Singapore Oil Imports: Iran Conflict Period (2024–2026)",
        y=0.97, x=0, xanchor="left",
        font=dict(size=15),
    ),
    xaxis_title="Month",
    template="plotly_white",
    hovermode="x unified",
    # Legend below the chart so it never overlaps the title
    legend=dict(
        orientation="h",
        yanchor="top", y=-0.30,
        xanchor="center", x=0.5,
    ),
    margin=dict(t=60, b=130),   # extra bottom margin for legend + event labels
)
fig.show()

## 5. Interpretation

The chart compares Singapore's Middle East oil imports with an alternative-supply proxy made from **Americas** and **Africa** imports.

If the alternative-supply line rises while Middle East imports decline, this suggests that Singapore may be diversifying away from Gulf/Hormuz-linked supply toward other exporters or routes, including US-linked supply and Africa/Cape-linked routes.

This does not directly prove individual ship rerouting through Houston or the Cape of Good Hope because the dataset does not include direct AIS route tracks. However, it provides useful trade-flow evidence for the broader project question: **how geopolitical conflict reshapes global oil trade routes**.
